In [ ]:
#!/usr/bin/env python
# -*- coding: utf-8 -*-

# Deep Neural Network

## Template 04
### Flowers Dataset — Custom CNN with Clean 224x224 Size Math

<img src='../../prasami_images/prasami_color_tutorials_small.png' width='400' alt="By Pramod Sharma : pramod.sharma@prasami.com"/>

In [ ]:
###-----------------
### Import libraries
###-----------------

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
import pathlib

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, f1_score
from tensorflow.keras import regularizers

from utils.helper import fn_plot_tf_hist, fn_plot_confusion_matrix, fn_plot_label

import tensorflow as tf

In [ ]:
###----------------------
### Some basic parameters
###----------------------

# ── Path config (auto-detects Colab vs local) ──
if 'COLAB_GPU' in os.environ or 'COLAB_RELEASE_TAG' in os.environ:
    inpDir = pathlib.Path('/root/.keras/datasets')   # Colab path
else:
    inpDir = Path(r'D:\Deep_Learning\SharedData')  # Local VS Code path

outDir   = Path('..') / 'output'
modelDir = Path('..') / 'models'
subDir   = 'flower_photos'
altName  = 'cnn_flowers'

RANDOM_STATE = 24
tf.random.set_seed(RANDOM_STATE)

TEST_SIZE   = 0.2
ALPHA       = 0.001
EPOCHS      = 50
PATIENCE    = 15
LR_PATIENCE = 7
FACTOR_LR   = 0.5
BATCH_SIZE  = 32

# ── Image size: 224x224 ──
# Rule: must be divisible by 2^5 = 32 (5 MaxPool layers)
# Max original image size in dataset = 240
# 224 is the largest clean size under 240
# Size flow: 224 → 112 → 56 → 28 → 14 → 7 (all even, no flooring loss)
IMG_HEIGHT = 224
IMG_WIDTH  = 224

# Plot decoration
params = {
    'legend.fontsize' : 'large',
    'figure.figsize'  : (15, 6),
    'axes.labelsize'  : 'x-large',
    'axes.titlesize'  : 'x-large',
    'xtick.labelsize' : 'large',
    'ytick.labelsize' : 'large',
}

CMAP = plt.cm.coolwarm
plt.rcParams.update(params)
plt.style.use('seaborn-v0_8-darkgrid')

## Basic Hygiene

In [ ]:
###------------------
### Memory Management
###------------------

physical_devices = tf.config.list_physical_devices('GPU')

if len(physical_devices) > 0:
    tf.config.experimental.set_memory_growth(physical_devices[0], True)
    print(physical_devices)
else:
    print('No GPU found — running on CPU')

## Load Dataset

Images are under flower_photos

     |- daisy
     |- dandelion
     |- roses
     |- sunflowers
     |- tulips

In [ ]:
# ── Download if on Colab ──
if 'COLAB_GPU' in os.environ or 'COLAB_RELEASE_TAG' in os.environ:
    import tarfile
    archive = inpDir / 'flower_photos_archive'
    extract_dir = inpDir / 'flower_photos'

    if not extract_dir.exists():
        if archive.exists():
            print("Extracting archive...")
            with tarfile.open(archive, 'r:gz') as tar:
                tar.extractall(inpDir)
        else:
            print("Downloading dataset...")
            url = "https://storage.googleapis.com/download.tensorflow.org/example_images/flower_photos.tgz"
            tf.keras.utils.get_file(origin=url, fname='flower_photos', untar=True)

data_dir = inpDir / subDir
print('Path  :', data_dir)
print('Exists:', data_dir.exists())

In [ ]:
# List only class directories
directories = [item for item in data_dir.iterdir() if item.is_dir()]
directories

In [ ]:
train_ds = tf.keras.utils.image_dataset_from_directory(
    data_dir,
    validation_split=TEST_SIZE,
    subset='training',
    seed=RANDOM_STATE,
    image_size=(IMG_HEIGHT, IMG_WIDTH),
    batch_size=BATCH_SIZE)

test_ds = tf.keras.utils.image_dataset_from_directory(
    data_dir,
    validation_split=TEST_SIZE,
    subset='validation',
    seed=RANDOM_STATE,
    image_size=(IMG_HEIGHT, IMG_WIDTH),
    batch_size=BATCH_SIZE)

In [ ]:
class_names = train_ds.class_names
num_classes = len(class_names)
print('Total classes:', num_classes, class_names)

In [ ]:
fn_plot_label(train_ds, test_ds)

In [ ]:
###-----------------------------
### Optimise pipeline performance
###-----------------------------

AUTOTUNE = tf.data.AUTOTUNE

train_ds = train_ds.cache().shuffle(1000).prefetch(buffer_size=AUTOTUNE)
test_ds  = test_ds.cache().prefetch(buffer_size=AUTOTUNE)

In [ ]:
plt.figure(figsize=(15, 8))

for images, labels in train_ds.take(1):
    for i in range(BATCH_SIZE):
        plt.subplot(int(BATCH_SIZE / 8), 8, i + 1)
        plt.grid(False)
        plt.imshow(images[i].numpy().astype('uint8'))
        plt.title(class_names[labels[i]])
        plt.axis('off')
    plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(15, 8))

for images, labels in test_ds.take(1):
    for i in range(BATCH_SIZE):
        plt.subplot(int(BATCH_SIZE / 8), 8, i + 1)
        plt.grid(False)
        plt.imshow(images[i].numpy().astype('uint8'))
        plt.title(class_names[labels[i]])
        plt.axis('off')
    plt.tight_layout()
plt.show()

In [ ]:
input_shape = (IMG_HEIGHT, IMG_WIDTH, 3)
print('Input shape:', input_shape)

## Define Model — Custom CNN

### Size Flow (224x224, 5 MaxPool layers, zero flooring)
```
Input:   224 x 224 x   3
Block 1: Conv(same) → 224 x 224 x  32 | MaxPool(2,2) → 112 x 112 x  32  ✅
Block 2: Conv(same) → 112 x 112 x  64 | MaxPool(2,2) →  56 x  56 x  64  ✅
Block 3: Conv(same) →  56 x  56 x 128 | MaxPool(2,2) →  28 x  28 x 128  ✅
Block 4: Conv(same) →  28 x  28 x 256 | MaxPool(2,2) →  14 x  14 x 256  ✅
Block 5: Conv(same) →  14 x  14 x 512 | MaxPool(2,2) →   7 x   7 x 512  ✅
Flatten: 7 x 7 x 512 = 25,088
```
- **No flooring** — 224 divisible by 2^5=32, every step perfectly even
- **No upscaling** — original images max 240px, 224 is pure downscale
- **Swish** activation in Conv (better than ReLU, no dying neurons)
- **ELU** activation in Dense (smooth gradients)
- **BatchNorm** in Conv blocks only (no Dropout conflict)
- **Dropout + L2** in Dense head only

In [ ]:
###--------------------------
### Dropout rates
### Only in Dense head — NOT in Conv blocks (avoids BatchNorm variance conflict)
###--------------------------
dor5 = 0.4   # after Dense 1  (512)
dor6 = 0.3   # after Dense 2  (256)
dor7 = 0.2   # after Dense 3  (128)

# L2 regularization — only on Dense layers
l2_reg = regularizers.l2(1e-4)

# Glorot (Xavier) uniform initializer
krnal_init = tf.keras.initializers.GlorotUniform(seed=RANDOM_STATE)

def build_model(input_shape, num_classes):

    # ── Data Augmentation ──
    # Added as FIRST layer inside model
    # Automatically OFF during model.predict() / model.evaluate()
    data_augmentation = tf.keras.Sequential([
        tf.keras.layers.RandomFlip("horizontal_and_vertical"),
        tf.keras.layers.RandomRotation(0.2),
        tf.keras.layers.RandomZoom(
            height_factor=(-0.2, 0.2),
            width_factor=(-0.2, 0.2)),
        tf.keras.layers.RandomTranslation(0.15, 0.15),
        tf.keras.layers.RandomContrast(0.2),
        tf.keras.layers.RandomBrightness(factor=(-0.2, 0.2)),
    ], name='data_augmentation')

    model = tf.keras.Sequential(name='cnn_flowers_v4')
    model.add(tf.keras.layers.Input(shape=input_shape))

    # Augmentation (training only)
    model.add(data_augmentation)

    # Normalize pixel values 0-255 → 0-1
    model.add(tf.keras.layers.Rescaling(1./255.))

    # ── Conv Block 1 ──
    # Conv(same): 224 x 224 x 32
    # MaxPool(2,2): 224/2 = 112 x 112 x 32  ✅ no flooring
    model.add(tf.keras.layers.Conv2D(32, (3,3),
                                     kernel_initializer=krnal_init,
                                     padding='same'))
    model.add(tf.keras.layers.BatchNormalization())
    model.add(tf.keras.layers.Activation('swish'))
    model.add(tf.keras.layers.MaxPool2D(pool_size=(2,2)))      # 112 x 112 x 32

    # ── Conv Block 2 ──
    # Conv(same): 112 x 112 x 64
    # MaxPool(2,2): 112/2 = 56 x 56 x 64  ✅ no flooring
    model.add(tf.keras.layers.Conv2D(64, (3,3),
                                     kernel_initializer=krnal_init,
                                     padding='same'))
    model.add(tf.keras.layers.BatchNormalization())
    model.add(tf.keras.layers.Activation('swish'))
    model.add(tf.keras.layers.MaxPool2D(pool_size=(2,2)))      # 56 x 56 x 64

    # ── Conv Block 3 ──
    # Conv(same): 56 x 56 x 128
    # MaxPool(2,2): 56/2 = 28 x 28 x 128  ✅ no flooring
    model.add(tf.keras.layers.Conv2D(128, (3,3),
                                     kernel_initializer=krnal_init,
                                     padding='same'))
    model.add(tf.keras.layers.BatchNormalization())
    model.add(tf.keras.layers.Activation('swish'))
    model.add(tf.keras.layers.MaxPool2D(pool_size=(2,2)))      # 28 x 28 x 128

    # ── Conv Block 4 ──
    # Conv(same): 28 x 28 x 256
    # MaxPool(2,2): 28/2 = 14 x 14 x 256  ✅ no flooring
    model.add(tf.keras.layers.Conv2D(256, (3,3),
                                     kernel_initializer=krnal_init,
                                     padding='same'))
    model.add(tf.keras.layers.BatchNormalization())
    model.add(tf.keras.layers.Activation('swish'))
    model.add(tf.keras.layers.MaxPool2D(pool_size=(2,2)))      # 14 x 14 x 256

    # ── Conv Block 5 ──
    # Conv(same): 14 x 14 x 512
    # MaxPool(2,2): 14/2 = 7 x 7 x 512  ✅ no flooring
    model.add(tf.keras.layers.Conv2D(512, (3,3),
                                     kernel_initializer=krnal_init,
                                     padding='same'))
    model.add(tf.keras.layers.BatchNormalization())
    model.add(tf.keras.layers.Activation('swish'))
    model.add(tf.keras.layers.MaxPool2D(pool_size=(2,2)))      # 7 x 7 x 512

    # ── Flatten ──
    # 7 x 7 x 512 = 25,088 features
    model.add(tf.keras.layers.Flatten())

    # ── Dense Head ──
    # L2 + Dropout only here (not in Conv — avoids BatchNorm conflict)

    # Dense 1: 25088 → 512
    model.add(tf.keras.layers.Dense(512,
                                    kernel_initializer=krnal_init,
                                    kernel_regularizer=l2_reg))
    model.add(tf.keras.layers.BatchNormalization())
    model.add(tf.keras.layers.Activation('elu'))
    model.add(tf.keras.layers.Dropout(rate=dor5))              # 0.4

    # Dense 2: 512 → 256
    model.add(tf.keras.layers.Dense(256,
                                    kernel_initializer=krnal_init,
                                    kernel_regularizer=l2_reg))
    model.add(tf.keras.layers.BatchNormalization())
    model.add(tf.keras.layers.Activation('elu'))
    model.add(tf.keras.layers.Dropout(rate=dor6))              # 0.3

    # Dense 3: 256 → 128
    model.add(tf.keras.layers.Dense(128,
                                    kernel_initializer=krnal_init,
                                    kernel_regularizer=l2_reg))
    model.add(tf.keras.layers.BatchNormalization())
    model.add(tf.keras.layers.Activation('elu'))
    model.add(tf.keras.layers.Dropout(rate=dor7))              # 0.2

    # Output — Softmax
    model.add(tf.keras.layers.Dense(num_classes,
                                    kernel_initializer=krnal_init,
                                    activation='softmax'))

    return model

model = build_model(input_shape, num_classes)
model.summary()

In [ ]:
loss_fn = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=False)

optimizer = tf.keras.optimizers.AdamW(
    learning_rate=ALPHA,
    weight_decay=1e-4
)

model.compile(optimizer=optimizer,
              loss=loss_fn,
              metrics=['accuracy'])

In [ ]:
checkpoint_filepath = modelDir / subDir / 'cnn_flowers_v4.weights.h5'

model_checkpoint_callback = tf.keras.callbacks.ModelCheckpoint(
    filepath=checkpoint_filepath,
    save_weights_only=True,
    monitor='val_loss',
    mode='auto',
    save_best_only=True
)

early_stopping_callback = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=PATIENCE,
    mode='auto',
    restore_best_weights=True
)

reduce_lr_callback = tf.keras.callbacks.ReduceLROnPlateau(
    monitor='val_loss',
    factor=FACTOR_LR,
    patience=LR_PATIENCE,
    min_lr=1e-6,
    verbose=1
)

## Training

In [ ]:
history = model.fit(
    train_ds,
    epochs=EPOCHS,
    validation_data=test_ds,
    verbose=1,
    callbacks=[early_stopping_callback,
               model_checkpoint_callback,
               reduce_lr_callback]
)

In [ ]:
loss_df = pd.DataFrame(history.history)
loss_df.head()

In [ ]:
fn_plot_tf_hist(loss_df)

## Evaluation — Test Dataset

In [ ]:
y_true, y_pred = [], []

for features, labels in test_ds:
    pred = model(features, training=False)
    pred = pred.numpy().argmax(axis=1)
    y_pred.append(pred)
    y_true.append(labels.numpy())

y_true = np.concatenate(y_true)
y_pred = np.concatenate(y_pred)

print("Test Accuracy :", accuracy_score(y_true, y_pred))
print("Test F1 Score :", f1_score(y_true, y_pred, average='weighted'))
print()
print(classification_report(y_true, y_pred, target_names=class_names))

In [ ]:
fn_plot_confusion_matrix(y_true, y_pred, labels=class_names)

## Evaluation — Train Dataset

In [ ]:
y_true_tr, y_pred_tr = [], []

for features, labels in train_ds:
    pred = model(features, training=False)
    pred = pred.numpy().argmax(axis=1)
    y_pred_tr.append(pred)
    y_true_tr.append(labels.numpy())

y_true_tr = np.concatenate(y_true_tr)
y_pred_tr = np.concatenate(y_pred_tr)

print("Train Accuracy :", accuracy_score(y_true_tr, y_pred_tr))
print("Train F1 Score :", f1_score(y_true_tr, y_pred_tr, average='weighted'))
print()
print(classification_report(y_true_tr, y_pred_tr, target_names=class_names))

In [ ]:
fn_plot_confusion_matrix(y_true_tr, y_pred_tr, labels=class_names)